# Gold drill-down — proving the hierarchy with SQL

A short, standalone notebook that queries the **Gold** layer directly to prove the network is
drillable up and down the **group ↔ company** hierarchy. Every step is plain SQL on `curated`
alone — `curated.edge` carries both the drill key (`focal_company_id`) and its label
(`focal_company_name`), so the drill needs no join back to a dimension.

Reads the built warehouse **read-only** — run `make pipeline` first.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import duckdb
from src import config

con = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
con.sql('SELECT COUNT(*) AS edges, COUNT(DISTINCT focal_group_id) AS groups FROM curated.edge').df()

,edges,groups
0,4110,13


## Pick a focal group with several direct companies
Drill is only interesting where a group has more than one legal entity underneath it, so we pick
the focal group with the most direct companies.

In [2]:
focal = con.sql('''
    SELECT focal_group_id
    FROM curated.edge
    GROUP BY 1
    ORDER BY COUNT(DISTINCT focal_company_id) DESC, SUM(gbp_volume) DESC
    LIMIT 1
''').fetchone()[0]
focal_name = con.sql(f"SELECT name FROM curated.node WHERE node_id = '{focal}'").fetchone()[0]
print('focal group:', focal, '-', focal_name)

focal group: 192230405343 - Koba2Maya


## 1. Group grain — the star-map view
One row per counterpart: money in/out of the focal **group**, in GBP, with count and fee. This is
exactly what the circles and diamonds on the star map show.

In [3]:
con.sql(f'''
    SELECT c.name AS counterpart,
           c.node_shape AS shape,
           e.direction,
           round(SUM(e.gbp_volume), 2) AS gbp_volume,
           SUM(e.txn_count)            AS txns,
           round(SUM(e.gbp_fee_revenue), 2) AS fee
    FROM curated.edge e
    JOIN curated.node c ON c.node_id = e.counterpart_id
    WHERE e.focal_group_id = '{focal}'
    GROUP BY 1, 2, 3
    ORDER BY gbp_volume DESC
    LIMIT 10
''').df()

,counterpart,shape,direction,gbp_volume,txns,fee
0,Onyx Trading,diamond,inflow,52541186.30,1.0,520.26
1,Nimbus Payments,diamond,inflow,6689824.56,2.0,0.00
2,Northwind Exchange,diamond,outflow,5828504.92,14.0,28.97
3,Onyx Capital,diamond,outflow,5693768.70,25.0,42.87
4,Atlas Technologies,diamond,inflow,3496325.81,21.0,9068.51
5,Marlin Global,diamond,outflow,2351570.80,4.0,6.82
6,Nimbus Exchange,diamond,outflow,2310671.67,4.0,9.30
7,Quantum Exchange,diamond,outflow,1682844.30,15.0,54.82
8,Beacon Financial,diamond,outflow,1564936.70,4.0,6904.22
9,Solstice Trading,diamond,outflow,1478827.49,31.0,44.96


## 2. Drill down — resolve the same group to its direct companies
Same focal group, now grouped by `focal_company_id`. The company **name** comes straight from
`curated.edge.focal_company_name` — no join to a dimension. This is the level **below** the group
circle: the entities that actually transacted, with the group's whole flow attributed across them.

In [4]:
con.sql(f'''
    SELECT focal_company_name AS direct_company,
           focal_company_id,
           round(SUM(gbp_volume), 2) AS gbp_volume,
           SUM(txn_count)            AS txns,
           round(SUM(gbp_fee_revenue), 2) AS fee
    FROM curated.edge
    WHERE focal_group_id = '{focal}'
    GROUP BY 1, 2
    ORDER BY gbp_volume DESC
''').df()

,direct_company,focal_company_id,gbp_volume,txns,fee
0,Koba2Maya Operations Ltd,262995992770,66235220.19,316.0,9270.28
1,Koba2Maya Operations Ltd,262958331091,18332531.33,47.0,549.01
2,Koba2Maya Group Holdings Ltd,262764306625,6719312.54,117.0,10767.81
3,Koba2Maya Holding S.à r.l.,262952169720,3209799.47,663.0,8540.69
4,Koba2Maya Holdings Ltd,262994262242,2909965.31,296.0,435.46
5,Koba2Maya Labs Ltd,262884422898,1491622.98,27.0,4232.29
6,Koba2Maya Group Holdings Ltd,262994262246,1483255.59,18.0,126.14
7,Koba2Maya Marketing LLC,262782728387,410007.16,61.0,141.20
8,Koba2Maya IP Ltd,262994262239,244112.23,56.0,465.25
9,Koba2Maya Brand Licensing Ltd,262958258410,181309.07,40.0,105.00


## 3. Drill down further — one company's own counterparts
Take the biggest direct company and show **its** network: who that specific entity transacted with.
This is the finest grain the model keeps.

In [5]:
row = con.sql(f'''
    SELECT focal_company_id, ANY_VALUE(focal_company_name) AS name
    FROM curated.edge WHERE focal_group_id = '{focal}'
    GROUP BY 1 ORDER BY SUM(gbp_volume) DESC LIMIT 1
''').fetchone()
company, company_name = row[0], row[1]
print('drilling into company:', company, '-', company_name)
con.sql(f'''
    SELECT cp.name AS counterpart, cp.node_shape AS shape, e.direction,
           round(SUM(e.gbp_volume), 2) AS gbp_volume, SUM(e.txn_count) AS txns
    FROM curated.edge e
    JOIN curated.node cp ON cp.node_id = e.counterpart_id
    WHERE e.focal_company_id = '{company}'
    GROUP BY 1, 2, 3 ORDER BY gbp_volume DESC LIMIT 10
''').df()

drilling into company: 262995992770 - Koba2Maya Operations Ltd


,counterpart,shape,direction,gbp_volume,txns
0,Onyx Trading,diamond,inflow,52541186.30,1.0
1,Northwind Exchange,diamond,outflow,5814412.73,9.0
2,Quantum Exchange,diamond,outflow,1679663.09,9.0
3,Cinder Payments,diamond,inflow,1146100.89,5.0
4,NXT Services,circle,outflow,1108786.11,6.0
5,Cobalt Solutions,diamond,outflow,766326.42,18.0
6,Cascade Partners,diamond,inflow,563746.11,3.0
7,Beacon Technologies,diamond,inflow,434773.96,3.0
8,Northwind Global,diamond,inflow,384371.19,12.0
9,Lumen Holdings,diamond,inflow,198871.18,5.0


## 4. Roll back up — the group total is exactly the sum of its companies
Conservation makes drill **lossless**: summing the company grain reproduces the group grain to the
penny. If this held only approximately, the hierarchy couldn't be trusted.

In [6]:
con.sql(f'''
    WITH group_grain AS (
        SELECT SUM(gbp_volume) AS v, SUM(txn_count) AS t
        FROM curated.edge WHERE focal_group_id = '{focal}'
    ),
    company_rollup AS (
        SELECT SUM(v) AS v, SUM(t) AS t FROM (
            SELECT focal_company_id, SUM(gbp_volume) AS v, SUM(txn_count) AS t
            FROM curated.edge WHERE focal_group_id = '{focal}' GROUP BY 1
        )
    )
    SELECT round(g.v, 2) AS group_volume,
           round(c.v, 2) AS company_rollup_volume,
           abs(g.v - c.v) < 0.01 AS volume_matches,
           g.t AS group_txns, c.t AS company_rollup_txns, g.t = c.t AS txns_match
    FROM group_grain g, company_rollup c
''').df()

,group_volume,company_rollup_volume,volume_matches,group_txns,company_rollup_txns,txns_match
0,1.017925e+08,1.017925e+08,True,1755.0,1755.0,True


## 5. The same drill works within any period
Slicing by month is orthogonal to the grain — here the biggest company's monthly volume. Swap
`focal_company_id` for `focal_group_id` to get the group-level monthly view.

In [7]:
con.sql(f'''
    SELECT e.month,
           round(SUM(e.gbp_volume), 2) AS gbp_volume,
           SUM(e.txn_count) AS txns
    FROM curated.edge e
    WHERE e.focal_company_id = '{company}'
    GROUP BY 1 ORDER BY 1
''').df()

,month,gbp_volume,txns
0,2025-07-01,6905082.37,56.0
1,2025-08-01,1052505.03,68.0
2,2025-09-01,56057638.09,49.0
3,2025-10-01,462857.58,48.0
4,2025-11-01,1181983.92,41.0
5,2025-12-01,575153.21,54.0


In [8]:
con.close()